In [1]:
import sqlite3

In [3]:
connection = sqlite3.connect('test.db')
cursor = connection.cursor()

In [ ]:
cursor.execute('''
    create table if not exists employees(
        id integer primary key autoincrement,
        name string,
        age integer,
        department string
    )
''')

connection.commit()

In [9]:
cursor.executemany('''
    insert into employees(name, age, department)
    values(?, ?, ?)
''', [
    ('Alice', 30, 'HR'),
    ('Bob', 25, 'IT'),
    ('Charlie', 35, 'Finance'),
    ('David', 28, 'Marketing'),
    ('Eva', 40, 'Sales')
])

connection.commit()

In [ ]:
cursor.execute('''
    select * from employees
''')
rows = cursor.fetchall()
for row in rows:
    print(row)

(1, 'Alice', 30, 'HR')
(2, 'Bob', 25, 'IT')
(3, 'Charlie', 35, 'Finance')
(4, 'David', 28, 'Marketing')
(5, 'Eva', 40, 'Sales')


In [13]:
cursor.execute('''
    update employees
    set age = 31
    where name = 'Alice'
''')
connection.commit()
cursor.execute('''
    select * from employees
''')
rows = cursor.fetchall()
for row in rows:
    print(row)

(1, 'Alice', 31, 'HR')
(2, 'Bob', 25, 'IT')
(3, 'Charlie', 35, 'Finance')
(4, 'David', 28, 'Marketing')
(5, 'Eva', 40, 'Sales')


In [14]:
cursor.execute('''
    delete from employees
    where name = 'Bob'
''')
connection.commit()
cursor.execute('''
    select * from employees
''')
rows = cursor.fetchall()
for row in rows:
    print(row)

(1, 'Alice', 31, 'HR')
(3, 'Charlie', 35, 'Finance')
(4, 'David', 28, 'Marketing')
(5, 'Eva', 40, 'Sales')


In [19]:
cursor.executemany('''
    insert into employees(name, age, department)
    values(?, ?, ?)
''', [
    ('Frank', 29, 'IT'),
    ('Grace', 33, 'HR')
])

In [23]:
cursor.execute('''
    select * from employees
''')
rows = cursor.fetchall()
for row in rows:
    print(row)

(1, 'Alice', 31, 'HR')
(3, 'Charlie', 35, 'Finance')
(4, 'David', 28, 'Marketing')
(5, 'Eva', 40, 'Sales')
(6, 'Frank', 29, 'IT')
(7, 'Grace', 33, 'HR')


In [24]:
cursor.execute('''
    select * from employees
    where age > 30
''')
rows = cursor.fetchall()
for row in rows:
    print(row)
    

(1, 'Alice', 31, 'HR')
(3, 'Charlie', 35, 'Finance')
(5, 'Eva', 40, 'Sales')
(7, 'Grace', 33, 'HR')


In [27]:
cursor.execute('''
    select * from employees
    where name like 'C%'
''')
rows = cursor.fetchall()
for row in rows:
    print(row)


(3, 'Charlie', 35, 'Finance')


In [28]:
cursor.execute('''
    create table if not exists departments(
        id integer primary key autoincrement,
        name string
    )
''')
connection.commit()
cursor.executemany('''
    insert into departments(name)
    values(?)
''', [
    ('HR',),
    ('IT',),
    ('Finance',),
    ('Marketing',),
    ('Sales',)
])
connection.commit()
cursor.execute('''
    select * from departments
''')
rows = cursor.fetchall()
for row in rows:
    print(row)

(1, 'HR')
(2, 'IT')
(3, 'Finance')
(4, 'Marketing')
(5, 'Sales')


In [29]:
# Rename the existing employees table
cursor.execute('''
    ALTER TABLE employees RENAME TO old_employees
''')

# Create a new employees table with the updated schema
cursor.execute('''
    CREATE TABLE employees(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name STRING,
        age INTEGER,
        department STRING,
        department_id INTEGER,
        FOREIGN KEY(department_id) REFERENCES departments(id)
    )
''')

# Copy data from the old employees table to the new one with department_id
cursor.execute('''
    INSERT INTO employees (id, name, age, department, department_id)
    SELECT old_employees.id, old_employees.name, old_employees.age, old_employees.department, departments.id
    FROM old_employees
    JOIN departments ON old_employees.department = departments.name
''')

# Drop the old employees table
cursor.execute('''
    DROP TABLE old_employees
''')

connection.commit()

# Verify the new schema
cursor.execute('''
    PRAGMA table_info(employees)
''')
rows = cursor.fetchall()
for row in rows:
    print(row)

(0, 'id', 'INTEGER', 0, None, 1)
(1, 'name', 'STRING', 0, None, 0)
(2, 'age', 'INTEGER', 0, None, 0)
(3, 'department', 'STRING', 0, None, 0)
(4, 'department_id', 'INTEGER', 0, None, 0)


In [30]:
cursor.execute('''
    select * from employees
''')
rows = cursor.fetchall()
for row in rows:
    print(row)

(1, 'Alice', 31, 'HR', 1)
(3, 'Charlie', 35, 'Finance', 3)
(4, 'David', 28, 'Marketing', 4)
(5, 'Eva', 40, 'Sales', 5)
(6, 'Frank', 29, 'IT', 2)
(7, 'Grace', 33, 'HR', 1)


In [31]:
def backup_database(source_db, backup_db):
    source_connection = sqlite3.connect(source_db)
    backup_connection = sqlite3.connect(backup_db)
    
    with backup_connection:
        source_connection.backup(backup_connection)
    
    source_connection.close()
    backup_connection.close()

# Call the function to back up the database
backup_database('test.db', 'backup.db')

In [32]:
def restore_database(source_db, backup_db):
    source_connection = sqlite3.connect(source_db)
    backup_connection = sqlite3.connect(backup_db)
    
    with source_connection:
        backup_connection.backup(source_connection)
    
    source_connection.close()
    backup_connection.close()

# Call the function to back up the database
restore_database('test.db', 'backup.db')

In [33]:
cursor.execute('''
    select * from employees
''')
rows = cursor.fetchall()
for row in rows:
    print(row)

(1, 'Alice', 31, 'HR', 1)
(3, 'Charlie', 35, 'Finance', 3)
(4, 'David', 28, 'Marketing', 4)
(5, 'Eva', 40, 'Sales', 5)
(6, 'Frank', 29, 'IT', 2)
(7, 'Grace', 33, 'HR', 1)
